# Text representation benchmark — off-market first

Compare **no text**, the existing **47-term TF-IDF**, **47-term binary Bag-of-Words**, and **three full 1,000-term approaches** on the same chronological folds with the same model hyperparameters.

- Full-vocabulary terms are selected **only from earlier training remarks**, without accessing sale prices or validation text.
- The *same vocabulary* is used for all three full-vocabulary representations within each fold.
- Historical comparable-sale features are computed once per fold and reused across experiments.
- This notebook runs only on earlier months (default September–November 2025). It **never** tunes on December 2025–May 2026.
- The old 47-term model and its default factory path remain unchanged.
- The 47 terms were historically selected using Elastic Net/Lasso; a separate supervised selection audit is appropriate if their original selection used validation labels.

Run from the repo root. No new Python libraries are required beyond the existing model dependencies.


In [1]:
# Locate the checkout from Jupyter/VS Code, including nested notebook folders.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("DATA_DASH_ROOT", Path.cwd())).expanduser().resolve()
ROOT = next((p for p in (_start, *_start.parents)
             if (p / "data.py").is_file() and (p / "models").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DATA_DASH_ROOT to your Data Dash checkout before running this notebook.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from features.text_features import TEXT_VARIANTS
from feature.text_benchmark import run_benchmark

BACKEND = "lightgbm"     # or "xgboost"; test one library at a time
TRACK = "off_market"     # list_aware supported too
MONTHS = ["2025-09", "2025-10", "2025-11"]
VARIANTS = list(TEXT_VARIANTS)
OUTPUT_DIR = Path("outputs/text_benchmark")

# Paste your *tuned* parameter overrides here for the fairest comparison.
# Leaving this empty uses repository defaults, which were NOT your Optuna winners.
PARAMS = {}

# Alternative: specify a JSON file containing your tuned params:
PARAMS_FILE = None  # e.g. Path("tuning/best_lightgbm_off_market.json")
if PARAMS_FILE is not None:
    PARAMS = json.loads(Path(PARAMS_FILE).read_text(encoding="utf-8"))

print("Backend:", BACKEND, "| Track:", TRACK)
print("Variants:", VARIANTS)
print("Model overrides:", PARAMS if PARAMS else "Repository defaults (not tuned)")


Backend: lightgbm | Track: off_market
Variants: ['no_text', 'selected_tfidf', 'selected_binary', 'full_tfidf', 'full_binary', 'full_count']
Model overrides: Repository defaults (not tuned)


In [2]:
pooled, monthly, predictions = run_benchmark(
    backend=BACKEND,
    track=TRACK,
    months=MONTHS,
    variants=VARIANTS,
    params=PARAMS,
    output_dir=OUTPUT_DIR,
    center_target=True,
)
display(pooled)



2025-09: 133,779 earlier sales -> 2,035 predictions
  no_text            MdAPE  9.330%  MAPE 12.116%
  selected_tfidf     MdAPE  8.614%  MAPE 10.857%
  selected_binary    MdAPE  8.527%  MAPE 10.860%
  full_tfidf         MdAPE  8.626%  MAPE 10.976%
  full_binary        MdAPE  8.605%  MAPE 10.874%
  full_count         MdAPE  8.641%  MAPE 10.937%

2025-10: 135,814 earlier sales -> 2,250 predictions
  no_text            MdAPE  8.853%  MAPE 11.714%
  selected_tfidf     MdAPE  8.114%  MAPE 10.646%
  selected_binary    MdAPE  7.956%  MAPE 10.562%
  full_tfidf         MdAPE  8.286%  MAPE 10.849%
  full_binary        MdAPE  8.102%  MAPE 10.681%
  full_count         MdAPE  8.127%  MAPE 10.656%

2025-11: 138,064 earlier sales -> 1,867 predictions
  no_text            MdAPE  9.710%  MAPE 12.247%
  selected_tfidf     MdAPE  8.451%  MAPE 11.170%
  selected_binary    MdAPE  8.469%  MAPE 11.107%
  full_tfidf         MdAPE  8.615%  MAPE 11.204%
  full_binary        MdAPE  8.611%  MAPE 11.131%
  full_c

,variant,homes,mape,mdape,median_signed
0,selected_binary,6152,10.826075,8.299618,1.547135
1,selected_tfidf,6152,10.875044,8.367723,1.627206
2,full_binary,6152,10.881242,8.403651,1.429097
3,full_count,6152,10.895008,8.429708,1.532097
4,full_tfidf,6152,10.998559,8.516121,1.410626
5,no_text,6152,12.008550,9.228172,1.460923


In [ ]:
display(monthly.pivot(index="month", columns="variant", values="mdape").round(3))
display(monthly.pivot(index="month", columns="variant", values="mape").round(3))

baseline = pooled.loc[pooled.variant.eq("selected_tfidf"), "mdape"]
if len(baseline):
    comparison = pooled.copy()
    comparison["mdape_minus_47_tfidf_pp"] = comparison["mdape"] - baseline.iloc[0]
    display(comparison.sort_values("mdape").round(3))

print("Saved paired property predictions in", OUTPUT_DIR / f"{BACKEND}_{TRACK}_predictions.csv")


## Next step (after development results)

If a candidate consistently helps on **September–November 2025**, freeze its vocabulary rule and model parameters, and change only the feature name in the usual `model_run.ipynb` `MODEL_SPECS`:

```python
MODEL_SPECS = [
    (LightGBMRegressor, "Xpt_spatiotemporal__full_binary", YOUR_TUNED_PARAMS)
]
```

Supported suffixes: `no_text`, `selected_tfidf`, `selected_binary`, `full_tfidf`, `full_binary`, `full_count`.

Then perform **one** December 2025–May 2026 confirmation for the shortlisted representation. The standalone benchmark saves all earlier development predictions and setup/vocabulary information for reproducibility; it does not alter `logs.md`, the production factory, or any existing saved model.
